# 00 · Descarga y exploración de SoccerNet-Calibration

**Objetivo:** descargar el dataset SoccerNet-Calibration, que contiene frames de partidos profesionales anotados con los keypoints de las líneas del campo, y explorar su estructura para entender el formato de las anotaciones.

**Contexto:** la detección de las líneas del campo (keypoints) es una tarea distinta a la detección de objetos. Es necesaria para calcular la homografía que proyecta las posiciones de los jugadores a una representación 2D del campo. En lugar de construir un dataset propio de keypoints (muy costoso de etiquetar), se utiliza SoccerNet-Calibration, dataset académico estándar con miles de frames ya anotados, sobre el que posteriormente se aplicará fine-tuning con frames propios.

**Entrada:** credenciales de acceso a SoccerNet (contraseña NDA).

**Salida:** dataset SoccerNet-Calibration descargado en `data/soccernet/` y comprensión de su estructura.

## 1. Instalación de la librería SoccerNet

In [1]:
import sys
!{sys.executable} -m pip install SoccerNet --quiet
print("Librería SoccerNet instalada correctamente")

Librería SoccerNet instalada correctamente



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración de la descarga

La contraseña de acceso se introduce de forma segura mediante `getpass` para no dejarla escrita en el notebook.

In [2]:
from SoccerNet.Downloader import SoccerNetDownloader
from getpass import getpass
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
SOCCERNET_DIR = REPO_ROOT / "data" / "soccernet"
SOCCERNET_DIR.mkdir(parents=True, exist_ok=True)

password = getpass("Introduce la contraseña de SoccerNet: ")

downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
downloader.password = password

print(f"Descargador configurado")
print(f"Directorio de descarga: {SOCCERNET_DIR}")

C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargador configurado
Directorio de descarga: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\soccernet


## 3. Descarga del dataset de calibración

Se descarga la tarea de calibración (`calibration`), que contiene los frames con sus anotaciones de keypoints del campo. La descarga incluye los conjuntos de entrenamiento, validación y test.

**Nota:** esta descarga puede pesar varios GB y tardar según la conexión. No utiliza GPU, por lo que puede ejecutarse en paralelo a otros entrenamientos.

In [3]:
downloader.downloadDataTask(
    task="calibration-2023",
    split=["train", "valid", "test"]
)

print("Descarga completada")

Descarga completada


## 4. Exploración de la estructura descargada

Se inspecciona qué se ha descargado y cómo está organizado.

In [4]:
import os

print("Estructura de SoccerNet-Calibration:\n")
for root, dirs, files in os.walk(SOCCERNET_DIR):
    nivel = root.replace(str(SOCCERNET_DIR), "").count(os.sep)
    indent = "  " * nivel
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} archivos en total)")

Estructura de SoccerNet-Calibration:

soccernet/
  calibration-2023/
    test.zip
    train.zip
    valid.zip


## 5. Descompresión de los archivos

El dataset se descarga comprimido en archivos ZIP (train, valid, test). Se descomprimen para acceder a los frames y sus anotaciones.

In [5]:
import zipfile

calibration_dir = SOCCERNET_DIR / "calibration-2023"
zips = ["train.zip", "valid.zip", "test.zip"]

for zip_name in zips:
    zip_path = calibration_dir / zip_name
    extract_dir = calibration_dir / zip_name.replace(".zip", "")

    if extract_dir.exists():
        print(f"{zip_name}: ya descomprimido, se omite")
        continue

    print(f"Descomprimiendo {zip_name}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print(f"  → {zip_name} descomprimido en {extract_dir.name}/")

print("\nDescompresión completada")

Descomprimiendo train.zip...
  → train.zip descomprimido en train/
Descomprimiendo valid.zip...
  → valid.zip descomprimido en valid/
Descomprimiendo test.zip...
  → test.zip descomprimido en test/

Descompresión completada


In [6]:
# Explorar la estructura tras descomprimir
print("Estructura tras descompresión:\n")

for split in ["train", "valid", "test"]:
    split_dir = calibration_dir / split
    if split_dir.exists():
        archivos = list(split_dir.iterdir())
        print(f"{split}/ ({len(archivos)} elementos)")
        for f in archivos[:8]:
            print(f"  {f.name}")
        if len(archivos) > 8:
            print(f"  ... y {len(archivos) - 8} más")
        print()

Estructura tras descompresión:

train/ (1 elementos)
  train

valid/ (1 elementos)
  valid

test/ (1 elementos)
  test



In [7]:
# Explorar el nivel anidado
print("Contenido real de cada split:\n")

for split in ["train", "valid", "test"]:
    inner_dir = calibration_dir / split / split
    if inner_dir.exists():
        archivos = sorted(inner_dir.iterdir())
        print(f"{split}/{split}/ ({len(archivos)} elementos)")
        for f in archivos[:10]:
            size = f.stat().st_size / 1024
            print(f"  {f.name} ({size:.1f} KB)")
        if len(archivos) > 10:
            print(f"  ... y {len(archivos) - 10} más")
        print()

Contenido real de cada split:

train/train/ (32928 elementos)
  00000.jpg (162.0 KB)
  00000.json (3.2 KB)
  00001.jpg (134.2 KB)
  00001.json (3.8 KB)
  00002.jpg (174.5 KB)
  00002.json (3.5 KB)
  00003.jpg (178.1 KB)
  00003.json (1.3 KB)
  00004.jpg (168.7 KB)
  00004.json (2.6 KB)
  ... y 32918 más

valid/valid/ (6426 elementos)
  00000.jpg (112.1 KB)
  00000.json (2.3 KB)
  00001.jpg (174.3 KB)
  00001.json (3.4 KB)
  00002.jpg (136.3 KB)
  00002.json (3.6 KB)
  00003.jpg (186.5 KB)
  00003.json (3.4 KB)
  00004.jpg (101.9 KB)
  00004.json (2.8 KB)
  ... y 6416 más

test/test/ (6284 elementos)
  00000.jpg (143.3 KB)
  00000.json (2.5 KB)
  00001.jpg (134.9 KB)
  00001.json (2.8 KB)
  00002.jpg (158.5 KB)
  00002.json (4.2 KB)
  00003.jpg (101.9 KB)
  00003.json (1.9 KB)
  00004.jpg (141.9 KB)
  00004.json (2.7 KB)
  ... y 6274 más



## 6. Inspección del formato de anotaciones

Se examina el contenido de un archivo JSON de anotación para entender cómo SoccerNet representa los keypoints del campo.

In [8]:
import json

train_inner = calibration_dir / "train" / "train"
ejemplo_json = train_inner / "00000.json"

with open(ejemplo_json, "r") as f:
    data = json.load(f)

print(f"Tipo de dato: {type(data)}")
print(f"\nNúmero de elementos/claves: {len(data)}")
print(f"\nContenido completo del JSON de ejemplo:\n")
print(json.dumps(data, indent=2)[:2000])

Tipo de dato: <class 'dict'>

Número de elementos/claves: 12

Contenido completo del JSON de ejemplo:

{
  "Circle left": [
    {
      "x": 0.6483906507492065,
      "y": 0.6289536952972412
    },
    {
      "x": 0.7058385014533997,
      "y": 0.6255833506584167
    },
    {
      "x": 0.8257812261581421,
      "y": 0.6132407188415527
    },
    {
      "x": 0.9097447991371155,
      "y": 0.5885555744171143
    },
    {
      "x": 0.9375208616256714,
      "y": 0.5739629864692688
    },
    {
      "x": 0.956458330154419,
      "y": 0.5571296215057373
    },
    {
      "x": 0.9652968645095825,
      "y": 0.5358055233955383
    },
    {
      "x": 0.9646666646003723,
      "y": 0.5245833396911621
    },
    {
      "x": 0.9400469064712524,
      "y": 0.5010092854499817
    }
  ],
  "Small rect. left top": [
    {
      "x": 0.5561770796775818,
      "y": 0.4116111099720001
    },
    {
      "x": 0.6732291579246521,
      "y": 0.4330092668533325
    }
  ],
  "Small rect. left main": 

In [9]:
print("Las 12 líneas/elementos anotados en este frame:\n")
for i, (key, puntos) in enumerate(data.items(), 1):
    print(f"{i:2}. {key}: {len(puntos)} puntos")

Las 12 líneas/elementos anotados en este frame:

 1. Circle left: 9 puntos
 2. Small rect. left top: 2 puntos
 3. Small rect. left main: 2 puntos
 4. Small rect. left bottom: 2 puntos
 5. Big rect. left top: 2 puntos
 6. Big rect. left main: 2 puntos
 7. Big rect. left bottom: 2 puntos
 8. Side line top: 2 puntos
 9. Side line left: 2 puntos
10. Goal left post right: 2 puntos
11. Goal left crossbar: 2 puntos
12. Goal left post left : 2 puntos


In [10]:
from collections import Counter

todas_las_lineas = Counter()
muestra = list(train_inner.glob("*.json"))[:500]

for json_path in muestra:
    with open(json_path) as f:
        d = json.load(f)
    for key in d.keys():
        todas_las_lineas[key] += 1

print(f"\nLíneas encontradas en una muestra de {len(muestra)} frames:\n")
for linea, count in todas_las_lineas.most_common():
    pct = 100 * count / len(muestra)
    print(f"  {linea:<30} aparece en {pct:.0f}% de los frames")


Líneas encontradas en una muestra de 500 frames:

  Side line top                  aparece en 58% de los frames
  Small rect. left main          aparece en 47% de los frames
  Side line left                 aparece en 45% de los frames
  Big rect. left main            aparece en 44% de los frames
  Big rect. right main           aparece en 42% de los frames
  Side line right                aparece en 42% de los frames
  Small rect. right main         aparece en 41% de los frames
  Circle left                    aparece en 40% de los frames
  Goal left crossbar             aparece en 40% de los frames
  Small rect. left top           aparece en 38% de los frames
  Goal left post right           aparece en 38% de los frames
  Big rect. right top            aparece en 37% de los frames
  Small rect. right top          aparece en 36% de los frames
  Circle right                   aparece en 36% de los frames
  Goal right crossbar            aparece en 34% de los frames
  Goal left post le